In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
import torch.utils.checkpoint as checkpoint
import gc

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Clear memory before training
torch.cuda.empty_cache()
gc.collect()

# Data Transforms (Advanced Augmentation)
transform = transforms.Compose([
    transforms.Resize((224, 224)),  
    transforms.RandomRotation(20),
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
    transforms.ToTensor(),  # Convert PIL Image → Tensor here
    transforms.RandomErasing(p=0.3, scale=(0.02, 0.2)),  # ✅ Should be after ToTensor()
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])


# Load Data
train_dir = "train"
valid_dir = "valid"

train_dataset = datasets.ImageFolder(train_dir, transform=transform)
valid_dataset = datasets.ImageFolder(valid_dir, transform=transform)

batch_size = 64  # **Increased batch size (adjust if VRAM runs out)**
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)
valid_loader = DataLoader(valid_dataset, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)

class_labels = train_dataset.classes
print(f"Classes: {class_labels}")

# **ResNet-50 Model with Checkpointing**
class CheckpointedResNet(nn.Module):
    def __init__(self, num_classes):
        super(CheckpointedResNet, self).__init__()
        self.model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)

        # **Freeze early layers and fine-tune last blocks**
        for param in self.model.parameters():
            param.requires_grad = False  # Freeze all layers initially
        for param in self.model.layer4.parameters():  
            param.requires_grad = True  # Unfreeze last block

        num_ftrs = self.model.fc.in_features
        self.model.fc = nn.Sequential(
            nn.Linear(num_ftrs, 512),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(512, num_classes),
            nn.Softmax(dim=1)
        )

    def forward(self, x):
        # **Apply checkpointing to memory-heavy layers**
        x = self.model.conv1(x)
        x = self.model.bn1(x)
        x = self.model.relu(x)
        x = self.model.maxpool(x)

        x = checkpoint.checkpoint(self.model.layer1, x)
        x = checkpoint.checkpoint(self.model.layer2, x)
        x = checkpoint.checkpoint(self.model.layer3, x)
        x = checkpoint.checkpoint(self.model.layer4, x)

        x = self.model.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.model.fc(x)
        return x

model = CheckpointedResNet(len(class_labels)).to(device)

# Loss and Optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# **Learning Rate Scheduler**
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=3, verbose=True)

# **Use Mixed Precision (AMP)**
scaler = torch.cuda.amp.GradScaler()

def free_memory():
    torch.cuda.empty_cache()
    gc.collect()

# Training Loop
epochs = 30  # **Reduced to 30 epochs**
early_stopping_patience = 7
best_val_acc = 0
early_stop_counter = 0

for epoch in range(epochs):
    model.train()
    running_loss = 0
    correct = 0
    total = 0
    
    for images, labels in train_loader:
        images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)
        optimizer.zero_grad()

        # **Use mixed precision for memory efficiency**
        with torch.cuda.amp.autocast():
            outputs = model(images)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

        # **Delete unused variables to free memory**
        del images, labels, outputs, loss
        torch.cuda.empty_cache()

    train_acc = 100 * correct / total
    print(f"Epoch {epoch+1}/{epochs}, Loss: {running_loss/len(train_loader):.4f}, Accuracy: {train_acc:.2f}%")

    # Validation
    model.eval()
    val_correct = 0
    val_total = 0
    with torch.no_grad():
        for images, labels in valid_loader:
            images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)
            outputs = model(images)
            _, predicted = outputs.max(1)
            val_total += labels.size(0)
            val_correct += predicted.eq(labels).sum().item()

            del images, labels, outputs  # Free memory after each batch
            torch.cuda.empty_cache()

    val_acc = 100 * val_correct / val_total
    print(f"Validation Accuracy: {val_acc:.2f}%")

    # **Update Learning Rate**
    scheduler.step(val_acc)

    # **Early Stopping**
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        early_stop_counter = 0
        torch.save(model.state_dict(), "best_model.pth")
        print("🔥 Best model saved!")
    else:
        early_stop_counter += 1

    if early_stop_counter >= early_stopping_patience:
        print("🛑 Early stopping triggered.")
        break

    free_memory()

# Load best model and evaluate
model.load_state_dict(torch.load("best_model.pth"))
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for images, labels in valid_loader:
        images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)
        outputs = model(images)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

        del images, labels, outputs
        torch.cuda.empty_cache()

print(f"🎯 Final Model Accuracy: {100 * correct / total:.2f}%")


Using device: cuda
Classes: ['Apple___Apple_scab', 'Apple___Black_rot', 'Apple___Cedar_apple_rust', 'Apple___healthy', 'Blueberry___healthy', 'Cherry_(including_sour)___Powdery_mildew', 'Cherry_(including_sour)___healthy', 'Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot', 'Corn_(maize)___Common_rust_', 'Corn_(maize)___Northern_Leaf_Blight', 'Corn_(maize)___healthy', 'Grape___Black_rot', 'Grape___Esca_(Black_Measles)', 'Grape___Leaf_blight_(Isariopsis_Leaf_Spot)', 'Grape___healthy', 'Orange___Haunglongbing_(Citrus_greening)', 'Peach___Bacterial_spot', 'Peach___healthy', 'Pepper,_bell___Bacterial_spot', 'Pepper,_bell___healthy', 'Potato___Early_blight', 'Potato___Late_blight', 'Potato___healthy', 'Raspberry___healthy', 'Soybean___healthy', 'Squash___Powdery_mildew', 'Strawberry___Leaf_scorch', 'Strawberry___healthy', 'Tomato___Bacterial_spot', 'Tomato___Early_blight', 'Tomato___Late_blight', 'Tomato___Leaf_Mold', 'Tomato___Septoria_leaf_spot', 'Tomato___Spider_mites Two-spotted_spide

/home/vani/Desktop/Projects/CropDiseaseDetection1345/.venv/lib/python3.12/site-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(
/tmp/ipykernel_110713/3057578693.py:93: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()
/tmp/ipykernel_110713/3057578693.py:116: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
/home/vani/Desktop/Projects/CropDiseaseDetection1345/.venv/lib/python3.12/site-packages/torch/_dynamo/eval_frame.py:745: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve th

Epoch 1/30, Loss: 3.0188, Accuracy: 69.20%
Validation Accuracy: 84.60%
🔥 Best model saved!
Epoch 2/30, Loss: 2.8534, Accuracy: 84.01%
Validation Accuracy: 87.68%
🔥 Best model saved!
Epoch 3/30, Loss: 2.8122, Accuracy: 87.85%
Validation Accuracy: 90.14%
🔥 Best model saved!
Epoch 4/30, Loss: 2.7934, Accuracy: 89.50%
Validation Accuracy: 91.94%
🔥 Best model saved!
Epoch 5/30, Loss: 2.7806, Accuracy: 90.68%
Validation Accuracy: 92.39%
🔥 Best model saved!
Epoch 6/30, Loss: 2.7759, Accuracy: 91.04%
Validation Accuracy: 92.08%
Epoch 7/30, Loss: 2.7658, Accuracy: 91.98%
Validation Accuracy: 94.30%
🔥 Best model saved!
Epoch 8/30, Loss: 2.7556, Accuracy: 92.94%
Validation Accuracy: 94.99%
🔥 Best model saved!
Epoch 9/30, Loss: 2.7520, Accuracy: 93.29%
Validation Accuracy: 94.74%
Epoch 10/30, Loss: 2.7519, Accuracy: 93.27%
Validation Accuracy: 95.18%
🔥 Best model saved!
Epoch 11/30, Loss: 2.7484, Accuracy: 93.62%
Validation Accuracy: 95.46%
🔥 Best model saved!
Epoch 12/30, Loss: 2.7460, Accuracy: 

In [19]:
import os

train_dir = "train"  # Adjust path if needed
class_labels = sorted(os.listdir(train_dir))  # Sorted for consistency

print("Trained classes:", class_labels)


Trained classes: ['Apple___Apple_scab', 'Apple___Black_rot', 'Apple___Cedar_apple_rust', 'Apple___healthy', 'Blueberry___healthy', 'Cherry_(including_sour)___Powdery_mildew', 'Cherry_(including_sour)___healthy', 'Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot', 'Corn_(maize)___Common_rust_', 'Corn_(maize)___Northern_Leaf_Blight', 'Corn_(maize)___healthy', 'Grape___Black_rot', 'Grape___Esca_(Black_Measles)', 'Grape___Leaf_blight_(Isariopsis_Leaf_Spot)', 'Grape___healthy', 'Orange___Haunglongbing_(Citrus_greening)', 'Peach___Bacterial_spot', 'Peach___healthy', 'Pepper,_bell___Bacterial_spot', 'Pepper,_bell___healthy', 'Potato___Early_blight', 'Potato___Late_blight', 'Potato___healthy', 'Raspberry___healthy', 'Soybean___healthy', 'Squash___Powdery_mildew', 'Strawberry___Leaf_scorch', 'Strawberry___healthy', 'Tomato___Bacterial_spot', 'Tomato___Early_blight', 'Tomato___Late_blight', 'Tomato___Leaf_Mold', 'Tomato___Septoria_leaf_spot', 'Tomato___Spider_mites Two-spotted_spider_mite', 'T

In [33]:
import torch
import torch.nn as nn
from torchvision import transforms, models
from PIL import Image
import json

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Define Class Labels
class_labels = [
    "Apple___Apple_scab", "Apple___Black_rot", "Apple___Cedar_apple_rust", "Apple___healthy", "Blueberry___healthy", 
    "Cherry_(including_sour)___Powdery_mildew", "Cherry_(including_sour)___healthy", "Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot", 
    "Corn_(maize)___Common_rust_", "Corn_(maize)___Northern_Leaf_Blight", "Corn_(maize)___healthy", "Grape___Black_rot", "Grape___Esca_(Black_Measles)", 
    "Grape___Leaf_blight_(Isariopsis_Leaf_Spot)", "Grape___healthy", "Orange___Haunglongbing_(Citrus_greening)", "Peach___Bacterial_spot", "Peach___healthy", 
    "Pepper,_bell___Bacterial_spot", "Pepper,_bell___healthy", "Potato___Early_blight", "Potato___Late_blight", "Potato___healthy", "Raspberry___healthy", 
    "Soybean___healthy", "Squash___Powdery_mildew", "Strawberry___Leaf_scorch", "Strawberry___healthy", "Tomato___Bacterial_spot", "Tomato___Early_blight", 
    "Tomato___Late_blight", "Tomato___Leaf_Mold", "Tomato___Septoria_leaf_spot", "Tomato___Spider_mites Two-spotted_spider_mite", "Tomato___Target_Spot", 
    "Tomato___Tomato_Yellow_Leaf_Curl_Virus", "Tomato___Tomato_mosaic_virus", "Tomato___healthy"
]

# Load the Trained Model
class PlantDiseaseModel(nn.Module):
    def __init__(self, num_classes):
        super(PlantDiseaseModel, self).__init__()
        self.model = models.resnet50(weights=None)
        num_ftrs = self.model.fc.in_features
        self.model.fc = nn.Sequential(
            nn.Linear(num_ftrs, 512),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(512, num_classes),
            nn.Softmax(dim=1)
        )
    
    def forward(self, x):
        return self.model(x)

# Instantiate Model and Load Weights
model = PlantDiseaseModel(len(class_labels)).to(device)
model.load_state_dict(torch.load("best_model.pth", map_location=device))
model.eval()

# Define Image Preprocessing Transform
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

def predict(image_path):
    """Function to classify an image using the trained model."""
    image = Image.open(image_path).convert("RGB")
    image = transform(image).unsqueeze(0).to(device)
    
    with torch.no_grad():
        outputs = model(image)
        _, predicted = torch.max(outputs, 1)
        class_name = class_labels[predicted.item()]
        confidence = torch.max(outputs).item()
    
    return {"class": class_name, "confidence": confidence}

# Example Usage
image_path = "test/test/TomatoYellowCurlVirus6.JPG"  # Replace with the path to your image
result = predict(image_path)
print(json.dumps(result, indent=4))


{
    "class": "Tomato___Tomato_Yellow_Leaf_Curl_Virus",
    "confidence": 1.0
}
